# Lab 01 · Better prompts and small evaluations

**Prepared by Mustafa Assaf** · 5–10 minute example

## Give the model the policy, then test it

**So far:** in Lab 00 the model answered without seeing the policy, and we compared one answer.

**New:** a *guided prompt* that contains the policy, and a small *evaluation* that checks the model on three cases.

A guided prompt has four labelled parts: the policy, one worked example, the case, and the exact reply format. JSON lets our code read the decision reliably. The one-sentence reason is for a person to check. Asking the model to check both criteria is a short, visible form of step-by-step reasoning. We do not ask for a long reasoning trace.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, save `FREEINFERENCE_API_KEY` in Colab Secrets and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"
from coursekit import make_ask
MODE = "practice"
ask = make_ask(mode=MODE)

## Step 1 · Build a guided prompt

`guided_prompt(case)` builds the complete prompt for any case. Printing it shows exactly what the model receives.

In [ ]:
import json
policy = "Choose microservices only if team_size >= 8 AND independent_deploys is true; otherwise choose modular monolith."
reply_format = '{"decision": "...", "reason": "one short sentence"}'
def guided_prompt(case):
    return (f"Policy: {policy}\nWorked example: team_size=2, independent_deploys=True -> modular monolith\n"
            f"Case: {case}\nCheck both criteria. Reply with only this JSON: {reply_format}")
print(guided_prompt({"team_size": 3, "independent_deploys": False}))

## Step 2 · Evaluate on three cases

An evaluation is a list of cases with their expected answers. These three are the course case, a case where both criteria hold, and a boundary case where only one holds. For each case, the code calls the model, reads the JSON with `json.loads`, and compares the decision with the expected answer.

Two different things are called "example" here. The *worked example* is part of the prompt, and the model reads it. The `example=` argument is the scripted practice reply, and no model ever sees it.

In [ ]:
tests = [(3, False, "modular monolith"),   # the course case
         (8, True, "microservices"),       # both criteria hold
         (8, False, "modular monolith")]   # boundary: only one criterion holds
passed = 0
for size, independent, expected in tests:
    case = {"team_size": size, "independent_deploys": independent}
    reply = ask(guided_prompt(case), example=json.dumps({"decision": expected, "reason": "Scripted practice reply."}))
    decision = json.loads(reply)["decision"].strip().lower()
    print(case, "->", decision, "| expected:", expected)
    passed += decision == expected  # True counts as 1
print(f"Passed {passed} of {len(tests)}")
assert passed == len(tests), "A case failed. Read its line above."

## What you should see

The four-part prompt, then three lines where the decision matches the expected answer, then `Passed 3 of 3`. Practice replies are scripted to be correct, so this score tests your evaluation code, not a model. Run live to test a real model. If a live reply is not plain JSON, `json.loads` raises an error. That is a format failure, so print `reply` to see what the model sent.

## Try it

Simulate a model that always picks microservices: in the `example=` reply, replace `expected` with `"microservices"`. Predict the score, then run Step 2. The evaluation finds the mistakes and the `assert` stops with an error. Put `expected` back, then add the case `(7, True, "modular monolith")`, which is one person below the threshold.

## Takeaway

Three cases test whether the model follows one simple policy. They say little about architecture quality. Read each `reason` too: a correct label with an invented reason is still a weak answer.

## Read further

- [Chain-of-thought prompting](https://arxiv.org/abs/2201.11903)
- [Prompt engineering](https://developers.openai.com/api/docs/guides/prompt-engineering)